
# PEAK M‑ATH — **Jittered Beaconing** (Full Framework + Advanced ML + Heatmaps)

This notebook extends a beaconing hunt to cover **C2 with irregular periodicity** (*jittered beaconing*), following the **PEAK** framework: **Prepare → Execute → Act → Knowledge**.

**Objectives**
- Detect beaconing communications **with jitter** (randomized intervals around a target period)
- Robust feature engineering (ACF, spectrum, dispersion, burstiness)
- **Ensemble ML**: Isolation Forest + One‑Class SVM + DBSCAN
- **Anomaly heatmaps** (by flow and over time)

> ⚠️ Replace Splunk connection parameters and adapt the SPL/datamodel to your environment.


In [ ]:
pass  # Placeholder (removed environment-specific output)

In [ ]:

# === Imports ===
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True


## PREPARE — Load SentinelOne CSV Data

In [ ]:

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'input' / 'sentinelone').exists() and (candidate / 'output').exists():
            return candidate
    return cwd

repo_root = find_repo_root()
input_pattern = repo_root / 'input' / 'sentinelone' / 'network-powerquery-results*.csv'
output_dir = repo_root / 'output' / 'network' / 'peak_beaconing_jittered'
output_dir.mkdir(parents=True, exist_ok=True)
input_files = sorted(repo_root.glob(str(Path('input/sentinelone/network-powerquery-results*.csv'))))

def _rel(p, root=repo_root):
    try:
        if hasattr(p, 'is_relative_to') and p.is_relative_to(root):
            return p.relative_to(root)
        return p
    except (ValueError, AttributeError):
        return p

if not input_files:
    raise FileNotFoundError(f'No input files found for pattern: {_rel(input_pattern)}')

dfs = [pd.read_csv(path) for path in input_files]
df = pd.concat(dfs, ignore_index=True)

rename_map = {
    'src.ip.address': 'src_ip',
    'dst.ip.address': 'dest_ip',
    'dst.port.number': 'dest_port',
    'event.time': '_time',
    'timestamp': '_time',
    'time': '_time',
    '@timestamp': '_time',
    'count': 'count'
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

required_base = ['src_ip', 'dest_ip', 'dest_port']
missing = [c for c in required_base if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns after normalization: {missing}')

if '_time' not in df.columns and len(df.columns) > 0:
    first_col = df.columns[0]
    first_col_dt = pd.to_datetime(df[first_col], errors='coerce', utc=True)
    valid_ratio = first_col_dt.notna().mean()
    if valid_ratio >= 0.9:
        df['_time'] = first_col_dt
        print(f"Using first column '{first_col}' as timestamp (_time). Parsed ratio: {valid_ratio:.1%}")

if '_time' not in df.columns:
    df['_time'] = pd.to_datetime('now', utc=True) + pd.to_timedelta(df.groupby(required_base).cumcount(), unit='s')
    print("Warning: no timestamp column detected. Generated synthetic timestamps per flow for processing.")
else:
    raw_time = df['_time'].astype(str).str.strip()
    normalized_time = (
        raw_time
        .str.replace('Â', '', regex=False)
        .str.replace('·', ' ', regex=False)
        .str.replace(r'\s+', ' ', regex=True)
        .str.replace(r'\b(am|pm)\b', lambda m: m.group(1).upper(), regex=True)
        .str.strip()
    )
    parsed_time = pd.to_datetime(normalized_time, format='%b %d %I:%M:%S.%f %p', errors='coerce', utc=True)
    fallback_mask = parsed_time.isna()
    if fallback_mask.any():
        parsed_time.loc[fallback_mask] = pd.to_datetime(normalized_time.loc[fallback_mask], errors='coerce', utc=True)
    if parsed_time.notna().any():
        current_year = pd.Timestamp.now('UTC').year
        parsed_time = parsed_time.where(parsed_time.dt.year != 1900, parsed_time + pd.DateOffset(years=current_year - 1900))
    df['_time'] = parsed_time

if 'count' not in df.columns:
    df['count'] = 1

df['dest_port'] = pd.to_numeric(df['dest_port'], errors='coerce')
df['count'] = pd.to_numeric(df['count'], errors='coerce').fillna(1)
df = df.dropna(subset=['_time', 'src_ip', 'dest_ip', 'dest_port'])

print(f'Loaded {len(df):,} rows from {len(input_files)} file(s).')
print(f'Input pattern: {_rel(input_pattern)}')
print(f'Output directory: {_rel(output_dir)}')
df.head()


## EXECUTE — Robust Feature Engineering for **jittered** beaconing

In [ ]:

if df.empty:
    raise ValueError("DataFrame is empty. Check SPL / permissions / time range.")

df = df.sort_values(["src_ip", "dest_ip", "dest_port", "_time"])
flow_cols = ["src_ip", "dest_ip", "dest_port"]

df["delta"] = df.groupby(flow_cols)["_time"].diff().dt.total_seconds()
df["delta"] = df["delta"].where(df["delta"] >= 0)

BIN_SEC = 10
MAX_LAG_SEC = 1800

from math import isfinite

def series_features(g):
    g = g.sort_values("_time")
    deltas = g["delta"].dropna()
    feats = {}
    n = len(deltas)
    feats["n_events"] = g.shape[0]
    feats["n_deltas"] = n
    if n < 3:
        for k in [
            "delta_mean","delta_median","delta_std","delta_mad","delta_iqr","delta_cv",
            "delta_min","delta_max","burstiness","acf_peak","spec_peak_ratio"
        ]:
            feats[k] = np.nan
        return feats

    q25, q75 = np.percentile(deltas, [25, 75])
    feats["delta_mean"]   = float(np.mean(deltas))
    feats["delta_median"] = float(np.median(deltas))
    feats["delta_std"]    = float(np.std(deltas))
    feats["delta_mad"]    = float(np.median(np.abs(deltas - feats["delta_median"])))
    feats["delta_iqr"]    = float(q75 - q25)
    feats["delta_cv"]     = float(feats["delta_std"] / (feats["delta_mean"] + 1e-9))
    feats["delta_min"]    = float(np.min(deltas))
    feats["delta_max"]    = float(np.max(deltas))

    mu, sigma = feats["delta_mean"], feats["delta_std"]
    feats["burstiness"] = float((sigma - mu) / (sigma + mu + 1e-9))

    ts = g.set_index("_time")
    counts = ts["count"] if "count" in ts.columns else pd.Series(1, index=ts.index)
    s = counts.resample(f"{BIN_SEC}s").sum().fillna(0)

    max_lag = int(MAX_LAG_SEC / BIN_SEC)
    x = s.values - s.values.mean()
    if x.std() == 0 or max_lag < 2:
        feats["acf_peak"] = 0.0
    else:
        acf = np.correlate(x, x, mode="full")
        acf = acf[acf.size // 2 :]
        acf = acf / (acf[0] + 1e-9)
        start = min(5, len(acf) - 1)
        end = min(max_lag, len(acf) - 1)
        feats["acf_peak"] = float(np.nanmax(acf[start:end])) if end > start else 0.0

    y = s.values
    if y.sum() == 0:
        feats["spec_peak_ratio"] = 0.0
    else:
        Y = np.fft.rfft(y)
        P = np.abs(Y)
        if len(P) <= 2:
            feats["spec_peak_ratio"] = 0.0
        else:
            peak = P[1:].max()
            feats["spec_peak_ratio"] = float(peak / (P[1:].sum() + 1e-9))

    return feats

rows = []
for key, g in df.groupby(flow_cols):
    feats = series_features(g)
    for k, v in zip(flow_cols, key):
        feats[k] = v
    rows.append(feats)

features = pd.DataFrame(rows)
features.head()


### EXECUTE — Advanced Modeling (IsolationForest + OneClassSVM + DBSCAN)

In [ ]:

model_cols = [c for c in features.columns if c not in ["src_ip","dest_ip","dest_port"]]
X = features[model_cols].fillna(0.0).values

scaler = StandardScaler()
Xz = scaler.fit_transform(X)

pca = PCA(n_components=min(8, Xz.shape[1]))
Xp = pca.fit_transform(Xz)

iso = IsolationForest(contamination=0.02, random_state=42)
iso.fit(Xz)
iso_score = -iso.decision_function(Xz)
iso_flag = (iso.predict(Xz) == -1).astype(int)

oc = OneClassSVM(nu=0.02, kernel='rbf', gamma='scale')
oc.fit(Xz)
oc_pred = oc.predict(Xz)
oc_flag = (oc_pred == -1).astype(int)
oc_raw = oc.decision_function(Xz)
oc_score = - (oc_raw - oc_raw.min()) / (oc_raw.max() - oc_raw.min() + 1e-9)

db = DBSCAN(eps=1.2, min_samples=5)
db_labels = db.fit_predict(Xp)
db_flag = (db_labels == -1).astype(int)
_db_score = db_flag.astype(float)

from sklearn.preprocessing import minmax_scale
iso_s = minmax_scale(iso_score)
oc_s  = minmax_scale(oc_score)
db_s  = minmax_scale(_db_score)
ensemble_score = (iso_s + oc_s + db_s) / 3.0

features_out = features.copy()
features_out['score_iso'] = iso_s
features_out['score_ocsvm'] = oc_s
features_out['score_dbscan'] = db_s
features_out['score_ensemble'] = ensemble_score
features_out['flag_anomaly'] = (ensemble_score >= 0.75).astype(int)

features_out.sort_values('score_ensemble', ascending=False).head(10)


## VISUALIZATIONS — Distributions, Scatter, **Anomaly Heatmaps**

In [ ]:

plt.figure(figsize=(10,5))
vals = df['delta'].dropna()
plt.hist(vals, bins=60, color='#4E79A7')
plt.title('Distribution of inter-arrival deltas (all communications)')
plt.xlabel('Delta (seconds)')
plt.ylabel('Frequency')
plt.show()


In [ ]:

plt.figure(figsize=(8,6))
sc = plt.scatter(features_out['acf_peak'], features_out['burstiness'], 
                 c=features_out['score_ensemble'], cmap='viridis', s=30)
plt.colorbar(sc, label='Anomaly score (ensemble)')
plt.xlabel('ACF peak (jitter-robust periodicity)')
plt.ylabel('Burstiness')
plt.title('Feature space colored by anomaly score')
plt.show()


In [ ]:

TOPK = 30
pair_scores = (features_out
               .groupby(['src_ip','dest_ip'])['score_ensemble']
               .max()
               .reset_index())

top_src = pair_scores.groupby('src_ip')['score_ensemble'].max().nlargest(TOPK).index
top_dst = pair_scores.groupby('dest_ip')['score_ensemble'].max().nlargest(TOPK).index

mat = (pair_scores[pair_scores['src_ip'].isin(top_src) & pair_scores['dest_ip'].isin(top_dst)]
       .pivot(index='src_ip', columns='dest_ip', values='score_ensemble').fillna(0))

plt.figure(figsize=(12,8))
sns.heatmap(mat, cmap='magma', vmin=0, vmax=1)
plt.title('Heatmap — Anomaly score per flow (max across ports)')
plt.xlabel('dest_ip')
plt.ylabel('src_ip')
plt.show()


In [ ]:

th = 0.75
anom_pairs = features_out[features_out['score_ensemble']>=th][['src_ip','dest_ip','dest_port']]

events = df.merge(anom_pairs, on=['src_ip','dest_ip','dest_port'], how='inner')
if not events.empty:
    events['minute'] = events['_time'].dt.floor('min')
    time_mat = (events.groupby(['minute','dest_ip']).size()
                .reset_index(name='n'))
    top_dest_t = time_mat.groupby('dest_ip')['n'].sum().nlargest(25).index
    piv = (time_mat[time_mat['dest_ip'].isin(top_dest_t)]
           .pivot(index='minute', columns='dest_ip', values='n').fillna(0))
    plt.figure(figsize=(14,6))
    sns.heatmap(piv.T, cmap='YlOrRd')
    plt.title('Temporal heatmap — Count of anomalous events per dest_ip (per minute)')
    plt.xlabel('Time (minute)')
    plt.ylabel('dest_ip')
    plt.show()
else:
    print("No anomalies above threshold for the temporal heatmap.")


## ACT — Export, SPL Pivots & Hunt Follow-up

In [ ]:

output_csv = output_dir / 'peak_beaconing_jittered_results.csv'
features_out.sort_values('score_ensemble', ascending=False).to_csv(output_csv, index=False)
print(f"Results exported → {_rel(output_csv)}")



**Useful SPL pivots** (adapt indexes/sourcetypes/datamodel to your env.)

```spl
| inputlookup output/network/peak_beaconing_jittered/peak_beaconing_jittered_results.csv 
| where score_ensemble>=0.75 
| table src_ip dest_ip dest_port score_ensemble
```

```spl
| tstats summariesonly=true count, values(All_Traffic.app) as app 
  FROM datamodel=Network_Traffic BY All_Traffic.src_ip, All_Traffic.dest_ip, All_Traffic.dest_port
| rename All_Traffic.* as *
| search src_ip IN (<suspect_src_ips>) dest_ip IN (<suspect_dest_ips>)
| sort - count
```

**PEAK tips — Jittered**
- Tune `BIN_SEC` and `MAX_LAG_SEC` to your time resolution and volume.
- Adjust thresholds: `contamination/nu`, decision threshold `0.75` for `score_ensemble`.
- Add **contextual enrichments** (ASN, URL categories, reputation, geo) for the Act phase.


## KNOWLEDGE — Continuous Improvement

- **Re-Add Topics to Backlog:** New jitter patterns, C2 frameworks, or evasion techniques discovered during analysis become future hunt hypotheses.
- **Communicate Findings:** Share confirmed jittered C2 channels with SOC leadership, incident response, and threat intelligence teams.
- **Feed Improvements Back:** Tune `BIN_SEC`, `MAX_LAG_SEC`, contamination/nu parameters, and ensemble thresholds based on confirmed true/false positive rates; add contextual enrichments (ASN, URL categories, reputation, geo) to improve future triage.
- **Measure Effectiveness:** Track ensemble score distributions, confirmed C2 detections, and false positive rates across runs to assess detection maturity.